# Customer Churn Prediction & Retention Analytics
### Comprehensive Machine Learning Modeling & EDA

**Project Objective:**
Customer attrition creates severe revenue loss for recurring subscription businesses. Retaining existing subscribers is 5–7x more cost-efficient than acquiring new customers. This notebook builds an end-to-end Machine Learning pipeline to identify at-risk customers, evaluate predictive models, uncover key churn drivers, and formulate retention strategies.

**Workflow Outline:**
1. Data Ingestion & Data Cleaning
2. Exploratory Data Analysis (EDA) & Behavioral Insights
3. Leak-Free Preprocessing Pipeline (`ColumnTransformer`)
4. Machine Learning Modeling (Logistic Regression, Random Forest, XGBoost)
5. Cross-Validation & Metric Evaluation (Accuracy, Recall, Precision, F1, ROC-AUC)
6. Model Explainability & Feature Importance
7. Business Retention Recommendations & Takeaways


In [ ]:
import os
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
)

sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams['figure.figsize'] = (10, 5)
warnings.filterwarnings('ignore')
print('All dependencies successfully imported!')


## 1. Data Ingestion & Cleaning
Load raw dataset from `data/customer_churn.csv`, verify column formats, and rectify data types.


In [ ]:
data_path = '../data/customer_churn.csv'
if not os.path.exists(data_path):
    data_path = 'data/customer_churn.csv'

df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(3)


In [ ]:
# Handling TotalCharges blank strings when tenure == 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.loc[df['tenure'] == 0, 'TotalCharges'] = 0.0

# Normalize SeniorCitizen to categorical string
df['SeniorCitizen'] = df['SeniorCitizen'].map({1: 'Yes', 0: 'No'}).fillna('No')

# Check missing values
print('Missing Values Check:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nTarget Class Balance:')
print(df['Churn'].value_counts(normalize=True))


## 2. Exploratory Data Analysis (EDA)
Analyze relationship between subscriber characteristics and churn propensity.


In [ ]:
# Overall Churn Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
churn_counts = df['Churn'].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%', colors=['#10B981', '#EF4444'], explode=[0, 0.05], startangle=90)
axes[0].set_title('Overall Churn Ratio')

sns.countplot(data=df, x='Churn', palette=['#10B981', '#EF4444'], ax=axes[1])
axes[1].set_title('Customer Volume by Churn Status')
plt.tight_layout()
plt.show()


In [ ]:
# Churn by Contract Type & Payment Method
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x='Contract', hue='Churn', palette=['#10B981', '#EF4444'], ax=axes[0])
axes[0].set_title('Churn by Contract Type')

sns.countplot(data=df, x='PaymentMethod', hue='Churn', palette=['#10B981', '#EF4444'], ax=axes[1])
axes[1].set_title('Churn by Payment Method')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# Tenure and Monthly Charges Distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.kdeplot(data=df, x='tenure', hue='Churn', common_norm=False, fill=True, palette=['#10B981', '#EF4444'], ax=axes[0])
axes[0].set_title('Tenure Density by Churn Outcome')

sns.boxplot(data=df, x='Churn', y='MonthlyCharges', palette=['#10B981', '#EF4444'], ax=axes[1])
axes[1].set_title('Monthly Charges by Churn Outcome')
plt.tight_layout()
plt.show()


In [ ]:
# Tech Support, Internet Service, and Online Security
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x='InternetService', hue='Churn', palette=['#10B981', '#EF4444'], ax=axes[0])
axes[0].set_title('Internet Service vs Churn')

sns.countplot(data=df, x='TechSupport', hue='Churn', palette=['#10B981', '#EF4444'], ax=axes[1])
axes[1].set_title('Tech Support vs Churn')

sns.countplot(data=df, x='OnlineSecurity', hue='Churn', palette=['#10B981', '#EF4444'], ax=axes[2])
axes[2].set_title('Online Security vs Churn')
plt.tight_layout()
plt.show()


## 3. Data Preprocessing & Leak-Free Pipeline
Use `ColumnTransformer` to apply standard scaling to numerical features and one-hot encoding to categorical features.


In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

X = df[num_cols + cat_cols].copy()
y = df['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

# Stratified 80/20 train/test split to prevent leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop=None, handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)
print(f'Training feature matrix shape: {X_train_trans.shape}')
print(f'Test feature matrix shape: {X_test_trans.shape}')


## 4. Model Training & 5-Fold Stratified Cross-Validation
Train and compare Logistic Regression, Random Forest, and XGBoost.


In [ ]:
pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=8, min_samples_split=10, class_weight='balanced', random_state=42),
    'XGBoost': XGBClassifier(n_estimators=120, max_depth=4, learning_rate=0.08, scale_pos_weight=pos_weight, eval_metric='logloss', random_state=42)
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    cv_res = cross_validate(model, X_train_trans, y_train, cv=cv, scoring=['accuracy', 'recall', 'f1', 'roc_auc'])
    model.fit(X_train_trans, y_train)
    y_pred = model.predict(X_test_trans)
    y_prob = model.predict_proba(X_test_trans)[:, 1]
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'CV_ROC_AUC_Mean': cv_res['test_roc_auc'].mean(),
        'CV_Recall_Mean': cv_res['test_recall'].mean()
    }

results_df = pd.DataFrame(results).T
results_df.round(4)


## 5. ROC Curves & Model Comparison


In [ ]:
plt.figure(figsize=(8, 6))
for name, model in models.items():
    y_prob = model.predict_proba(X_test_trans)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.4f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve Comparison Across Models')
plt.legend(loc='lower right')
plt.show()


## 6. Feature Importance & Interpretability


In [ ]:
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
feature_names = num_cols + cat_encoder.get_feature_names_out(cat_cols).tolist()

rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.head(15).values, y=importances.head(15).index, palette='Blues_r')
plt.title('Top 15 Predictive Features (Random Forest)')
plt.xlabel('Feature Importance')
plt.show()


## 7. Conclusions & Strategic Retention Actions
1. **Contract Type:** Month-to-month contracts are the number one predictor of churn. Long-term (1- or 2-year) commitments anchor customer loyalty.
2. **Tenure Lifecycles:** Churn is heavily front-loaded in months 1 to 6. Early-stage onboarding programs are critical.
3. **Value Bundling:** Customers without Tech Support or Online Security churn at twice the rate. Offering introductory trials of value-added security boosts stickiness.
4. **Autopay Incentives:** Customers paying via electronic check churn significantly more than those on automated ACH/credit card autopay.
